# Validação dos dados MovieLens

## Objetivo

Validar schema, integridade, cardinalidade e qualidade dos arquivos brutos `movies`, `ratings`, `tags` e `links`.

Este notebook é somente diagnóstico: ele não altera nem gera dados processados.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## Localização dos dados

A busca pela raiz torna o notebook executável tanto a partir de `notebooks/` quanto da raiz do projeto.

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Não foi possível localizar data/raw.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

DATA_PATHS = {
    "movies": RAW_DATA_DIR / "movies.csv",
    "ratings": RAW_DATA_DIR / "ratings.csv",
    "tags": RAW_DATA_DIR / "tags.csv",
    "links": RAW_DATA_DIR / "links.csv",
}

missing_files = [str(path) for path in DATA_PATHS.values() if not path.exists()]
assert not missing_files, f"Arquivos ausentes: {missing_files}"

DATA_PATHS

{'movies': PosixPath('/home/mkiku/ml-practice/MovieLens_project/data/raw/movies.csv'),
 'ratings': PosixPath('/home/mkiku/ml-practice/MovieLens_project/data/raw/ratings.csv'),
 'tags': PosixPath('/home/mkiku/ml-practice/MovieLens_project/data/raw/tags.csv'),
 'links': PosixPath('/home/mkiku/ml-practice/MovieLens_project/data/raw/links.csv')}

## Carregamento

In [3]:
movies = pd.read_csv(DATA_PATHS["movies"])
ratings = pd.read_csv(DATA_PATHS["ratings"])
tags = pd.read_csv(DATA_PATHS["tags"])
links = pd.read_csv(DATA_PATHS["links"])

datasets = {
    "movies": movies,
    "ratings": ratings,
    "tags": tags,
    "links": links,
}

pd.DataFrame(
    {
        "linhas": {name: len(frame) for name, frame in datasets.items()},
        "colunas": {name: frame.shape[1] for name, frame in datasets.items()},
    }
)

,linhas,colunas
movies,9742,3
ratings,100836,4
tags,3683,4
links,9742,3


## Resumo dos schemas

A tabela abaixo concentra tipo, nulos e cardinalidade para evitar várias inspeções repetidas.

In [4]:
def summarize_dataframe(frame: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "dtype": frame.dtypes.astype(str),
            "non_null": frame.notna().sum(),
            "null": frame.isna().sum(),
            "null_pct": frame.isna().mean().mul(100).round(2),
            "unique": frame.nunique(dropna=True),
        }
    )


for name, frame in datasets.items():
    print(f"\n{name.upper()} — {frame.shape[0]:,} linhas")
    display(frame.head())
    display(summarize_dataframe(frame))


MOVIES — 9,742 linhas


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,dtype,non_null,null,null_pct,unique
movieId,int64,9742,0,0.00,9742
title,object,9742,0,0.00,9737
genres,object,9742,0,0.00,951



RATINGS — 100,836 linhas


,userId,movieId,rating,timestamp
0,1,1,4.00,964982703
1,1,3,4.00,964981247
2,1,6,4.00,964982224
3,1,47,5.00,964983815
4,1,50,5.00,964982931


...

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


,dtype,non_null,null,null_pct,unique
userId,int64,3683,0,0.00,58
movieId,int64,3683,0,0.00,1572
tag,object,3683,0,0.00,1589
timestamp,int64,3683,0,0.00,3411



LINKS — 9,742 linhas


,movieId,imdbId,tmdbId
0,1,114709,862.00
1,2,113497,"8,844.00"
2,3,113228,"15,602.00"
3,4,114885,"31,357.00"
4,5,113041,"11,862.00"


,dtype,non_null,null,null_pct,unique
movieId,int64,9742,0,0.00,9742
imdbId,int64,9742,0,0.00,9742
tmdbId,float64,9734,8,0.08,9733


## Validação dos schemas

Estas verificações falham de forma explícita se os arquivos mudarem de formato.

In [5]:
expected_columns = {
    "movies": {"movieId", "title", "genres"},
    "ratings": {"userId", "movieId", "rating", "timestamp"},
    "tags": {"userId", "movieId", "tag", "timestamp"},
    "links": {"movieId", "imdbId", "tmdbId"},
}

for name, expected in expected_columns.items():
    actual = set(datasets[name].columns)
    assert actual == expected, f"Schema inválido em {name}: {actual}"

print("Schemas válidos.")

Schemas válidos.


## Integridade das chaves e valores

In [6]:
validation_checks = {
    "movieId único em movies": movies["movieId"].is_unique,
    "movieId único em links": links["movieId"].is_unique,
    "ratings sem chaves nulas": ratings[["userId", "movieId"]].notna().all().all(),
    "ratings entre 0.5 e 5.0": ratings["rating"].between(0.5, 5.0).all(),
    "ratings referenciam movies": set(ratings["movieId"]).issubset(set(movies["movieId"])),
    "tags referenciam movies": set(tags["movieId"]).issubset(set(movies["movieId"])),
    "links referenciam movies": set(links["movieId"]).issubset(set(movies["movieId"])),
}

validation_results = pd.DataFrame.from_dict(
    validation_checks, orient="index", columns=["passed"]
)
display(validation_results)
assert validation_results["passed"].all(), "Uma ou mais validações falharam."

,passed
movieId único em movies,True
movieId único em links,True
ratings sem chaves nulas,True
ratings entre 0.5 e 5.0,True
ratings referenciam movies,True
tags referenciam movies,True
links referenciam movies,True


## Duplicatas

Uma interação é identificada pelo trio usuário, filme e timestamp. Tags idênticas também são verificadas no mesmo instante.

In [7]:
duplicate_summary = pd.Series(
    {
        "movies.movieId": movies.duplicated(subset=["movieId"]).sum(),
        "ratings.user_movie_timestamp": ratings.duplicated(
            subset=["userId", "movieId", "timestamp"]
        ).sum(),
        "tags.user_movie_tag_timestamp": tags.duplicated(
            subset=["userId", "movieId", "tag", "timestamp"]
        ).sum(),
        "links.movieId": links.duplicated(subset=["movieId"]).sum(),
    },
    name="duplicate_rows",
)
duplicate_summary.to_frame()

,duplicate_rows
movies.movieId,0
ratings.user_movie_timestamp,0
tags.user_movie_tag_timestamp,0
links.movieId,0


## Cobertura do catálogo e dos links

In [8]:
rated_movie_ids = set(ratings["movieId"])
catalog_movie_ids = set(movies["movieId"])

coverage_summary = pd.Series(
    {
        "catalog_movies": len(catalog_movie_ids),
        "rated_movies": len(rated_movie_ids),
        "unrated_catalog_movies": len(catalog_movie_ids - rated_movie_ids),
        "movies_with_tags": tags["movieId"].nunique(),
        "missing_imdb_id": links["imdbId"].isna().sum(),
        "missing_tmdb_id": links["tmdbId"].isna().sum(),
    },
    name="value",
)
coverage_summary.to_frame()

,value
catalog_movies,9742
rated_movies,9724
unrated_catalog_movies,18
movies_with_tags,1572
missing_imdb_id,0
missing_tmdb_id,8


## Resumo para a próxima etapa

- Os quatro arquivos foram tratados como fontes independentes e tiveram seus schemas validados.
- Ratings são feedback explícito; notas iguais ou maiores que 4 poderão ser tratadas como relevantes nas métricas Top-K.
- Tags possuem relação muitos-para-um com filmes. Elas devem ser agregadas por `movieId` antes de serem unidas ao catálogo.
- A próxima etapa deve estudar sparsidade, cauda longa, atividade dos usuários, popularidade e comportamento temporal.